# Customer Simulator Fine-Tuning (Gemma 4 4B, QLoRA)

Fine-tunes a small language model to simulate customer responses given conversation history + agent action.  
The trained simulator will be used as the **environment** for a contextual-bandit / RL customer-support agent.

**Model served via Ollama** — `gemma4:e4b`  
**Dataset** — ABCD (Action-Based Conversations Dataset)

---
## Format
```
[HISTORY]
<turn-by-turn dialogue so far>
[AGENT ACTION]
<latest agent utterance / system action>
→ model predicts next customer utterance
```

## 0. Install dependencies

In [1]:
# Run once; restart kernel afterwards if needed
import subprocess, sys

packages = [
    "torch",
    "transformers>=4.40",
    "peft>=0.10",
    "bitsandbytes>=0.43",
    "datasets>=2.18",
    "accelerate>=0.28",
    "trl>=0.8",
    "ollama",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("Dependencies installed.")

Dependencies installed.


## 1. Explore the ABCD Dataset

In [2]:
import json
import os
import random
from pathlib import Path

# Relative path — works on any machine
DATASET_PATH = Path("abcd_v1.1.json")

with open(DATASET_PATH, "r") as f:
    raw = json.load(f)

splits = {k: v for k, v in raw.items()}
print("Splits:", {k: len(v) for k, v in splits.items()})
print("Total conversations:", sum(len(v) for v in splits.values()))

Splits: {'train': 8034, 'dev': 1004, 'test': 1004}
Total conversations: 10042


In [3]:
# Inspect a single conversation
sample = splits["train"][0]
print("Conversation ID  :", sample["convo_id"])
print("Flow             :", sample["scenario"]["flow"])
print("Subflow          :", sample["scenario"]["subflow"])
print()
print("--- First 10 turns (speaker, text) ---")
for turn in sample["original"][:10]:
    speaker, text = turn[0], turn[1]
    print(f"  [{speaker:8s}] {text}")

print()
print("Keys in each conversation:", list(sample.keys()))
print("Scenario keys           :", list(sample["scenario"].keys()))

Conversation ID  : 3592
Flow             : product_defect
Subflow          : return_size

--- First 10 turns (speaker, text) ---
  [agent   ] Hi!
  [agent   ] How can I help you?
  [customer] Hi! I need to return an item, can you help me with that?
  [agent   ] sure, may I have your name please?
  [customer] Crystal Minh
  [agent   ] thanks, may I ask the reason for the return?
  [action  ] Account has been pulled up for Crystal Minh.
  [customer] I got the wrong size.
  [agent   ] ok, may I have your username, email address and order ID please?
  [customer] Username: cminh730

Keys in each conversation: ['convo_id', 'scenario', 'original', 'delexed']
Scenario keys           : ['personal', 'order', 'product', 'flow', 'subflow']


## 2. Build the Training Dataset

For every **customer** turn (after turn 1), we create one training example:  
- **input** = conversation history up to (but not including) the customer's turn, ending with the last agent/action turn  
- **output** = that customer utterance

In [4]:
def build_samples(conversations):
    """
    For each conversation, slide a window: every customer turn becomes
    one (input, output) sample where:
      input  = [HISTORY] ... [AGENT ACTION] <last agent/action text>
      output = next customer utterance
    """
    samples = []
    for convo in conversations:
        turns = convo["original"]  # list of [speaker, text]
        history_parts = []

        for i, turn in enumerate(turns):
            speaker, text = turn[0], turn[1]

            if speaker == "customer" and len(history_parts) > 0:
                # Find the last agent or action turn to label as [AGENT ACTION]
                # Everything before it is [HISTORY]
                last_agent_idx = None
                for j in range(len(history_parts) - 1, -1, -1):
                    if history_parts[j]["role"] in ("agent", "action"):
                        last_agent_idx = j
                        break

                if last_agent_idx is None:
                    # No agent action yet — skip
                    history_parts.append({"role": speaker, "text": text})
                    continue

                # Build history string (everything up to last_agent_idx - 1)
                hist_lines = []
                for hp in history_parts[:last_agent_idx]:
                    role_tag = "Agent" if hp["role"] == "agent" else \
                               "Action" if hp["role"] == "action" else "Customer"
                    hist_lines.append(f"{role_tag}: {hp['text']}")

                # Build the agent action string (may span consecutive agent/action turns)
                agent_lines = []
                for hp in history_parts[last_agent_idx:]:
                    role_tag = "Agent" if hp["role"] == "agent" else "Action"
                    agent_lines.append(f"{role_tag}: {hp['text']}")

                history_str = "\n".join(hist_lines) if hist_lines else "(conversation start)"
                agent_str   = "\n".join(agent_lines)

                input_text = (
                    f"[HISTORY]\n{history_str}\n"
                    f"[AGENT ACTION]\n{agent_str}"
                )
                output_text = text.strip()

                samples.append({"input": input_text, "output": output_text})

            history_parts.append({"role": speaker, "text": text})

    return samples


# Use all three splits (model hasn't seen them; only the simulator trains here)
all_convos = splits["train"] + splits["dev"] + splits["test"]
all_samples = build_samples(all_convos)

print(f"Total usable samples: {len(all_samples)}")
print()
print("=== Sample preview ===")
ex = all_samples[42]
print("INPUT:")
print(ex["input"])
print("\nOUTPUT:")
print(ex["output"])

Total usable samples: 87695

=== Sample preview ===
INPUT:
[HISTORY]
Agent: Hello, how can i help you
Customer: Hello. I have a really cool party coming up. And I need some new clothes ASAP. I am thinking of ordering them to come by overnight shipping
Customer: Do you know how much that costs?
Action: Searching the FAQ pages ...
Action: System Action: search pricing
[AGENT ACTION]
Agent: You want to know how much overnight shipping is, correct?
Action: yes
Action: like how much it costs
Action: I imagine that there is an extra fee

OUTPUT:
i want these clothes quickly


## 3. Estimate Safe Sample Count for RTX 4050

In [ ]:
import torch

VRAM_GB = 6          # RTX 4050 laptop — 6 GB
BYTES_PER_TOKEN  = 2 # 4-bit quant ≈ 2 bytes/param effective overhead per token in activations
SEQ_LEN = 512        # training sequence length
BATCH   = 1          # micro-batch

# Rough heuristic: 4B model in 4-bit ≈ 2 GB weights; ~3 GB left for activations
# activation mem per sample ≈ SEQ_LEN * hidden * layers * bytes, very rough 60 MB/sample at 512
# Safe ceiling derived empirically for Gemma 4B family
MAX_SAFE_SAMPLES = 8000   # memory per epoch is independent of dataset size (SGD streams)

total_N = len(all_samples)
print(f"Total samples (N)          : {total_N}")
print(f"Recommended max (M)        : {MAX_SAFE_SAMPLES}")
print(f"GPU available              : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                        : {torch.cuda.get_device_name(0)}")
    free, tot = torch.cuda.mem_get_info()
    print(f"Free VRAM                  : {free/1e9:.1f} GB / {tot/1e9:.1f} GB")

print()
print(f"Tip: For a quick sanity-check run use 500–1000 samples.")
print(f"     For a proper fine-tune use up to {MAX_SAFE_SAMPLES} samples.")

: 

In [ ]:
# ── USER CHOICE ──────────────────────────────────────────────────────────────
# Set SUBSET_SIZE to any value <= MAX_SAFE_SAMPLES (or <= total_N)
# Smaller = faster training, less coverage
# Larger  = more faithful simulator, longer training

SUBSET_SIZE = 3000   # <-- change this (recommended: 500 for quick test, 3000-8000 for full)

# ─────────────────────────────────────────────────────────────────────────────
assert SUBSET_SIZE <= total_N, f"SUBSET_SIZE {SUBSET_SIZE} > total samples {total_N}"
assert SUBSET_SIZE <= MAX_SAFE_SAMPLES, (
    f"SUBSET_SIZE {SUBSET_SIZE} may exceed safe VRAM limit {MAX_SAFE_SAMPLES}. "
    "Lower it or proceed at your own risk."
)

random.seed(42)
subset = random.sample(all_samples, SUBSET_SIZE)
print(f"Using {len(subset)} samples for fine-tuning.")

## 4. Tokenise & Format

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
BASE_MODEL   = "google/gemma-4-E4B"   #(Ollama gemma4:e4b ≈ 4B)

ADAPTER_DIR  = "./gemma_simulator_lora"  # where LoRA weights are saved

MAX_SEQ_LEN  = 512    # tokens per sample (reduce to 256 to save more VRAM)
BATCH_SIZE   = 2      # micro-batch per GPU step
GRAD_ACCUM   = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS       = 2      # number of passes over SUBSET_SIZE samples
LR           = 2e-4   # learning rate
# ─────────────────────────────────────────────────────────────────────────────

print("Config set. Proceeding to load model...")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

# 4-bit quantisation config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokeniser from {BASE_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model (4-bit) ...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model loaded.")

In [ ]:
# Attach LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Gemma attention layers
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Build prompt string: input → output as a single causal sequence
# The model is trained to predict the output tokens; input tokens are masked (labels = -100)

def make_prompt(sample):
    return f"{sample['input']}\n[CUSTOMER RESPONSE]\n{sample['output']}{tokenizer.eos_token}"

def tokenise(sample):
    prompt     = make_prompt(sample)
    input_only = f"{sample['input']}\n[CUSTOMER RESPONSE]\n"

    full_ids   = tokenizer(prompt,      max_length=MAX_SEQ_LEN, truncation=True, padding="max_length")
    prefix_ids = tokenizer(input_only,  add_special_tokens=False)["input_ids"]
    prefix_len = len(prefix_ids)

    labels = full_ids["input_ids"].copy()
    # Mask the input / history tokens — only train on the customer utterance
    for i in range(min(prefix_len, len(labels))):
        labels[i] = -100
    # Mask padding
    for i, (tok, msk) in enumerate(zip(full_ids["input_ids"], full_ids["attention_mask"])):
        if msk == 0:
            labels[i] = -100

    return {
        "input_ids":      full_ids["input_ids"],
        "attention_mask": full_ids["attention_mask"],
        "labels":         labels,
    }

hf_dataset = Dataset.from_list(subset)
tokenised  = hf_dataset.map(tokenise, remove_columns=["input", "output"])
tokenised.set_format("torch")
print(f"Dataset ready: {len(tokenised)} samples, columns: {tokenised.column_names}")

## 5. Fine-Tune

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True, pad_to_multiple_of=8
    ),
)

print("Starting training...")
result = trainer.train()
print(f"\nTraining complete.")
print(f"  Final loss   : {result.training_loss:.4f}")
print(f"  Total steps  : {result.global_step}")

## 6. Save the LoRA Adapter

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to: {ADAPTER_DIR}")
print("Files:", os.listdir(ADAPTER_DIR))

## 7. Inference — Test the Simulator

Two paths available:
- **Path A** — Load the saved LoRA adapter directly (requires the HF base model)  
- **Path B** — Use Ollama (`gemma4:e4b`) for quick interactive testing without the adapter

### 7A. HuggingFace LoRA Inference

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

def load_simulator(base_model_id: str, adapter_dir: str):
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(adapter_dir)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb_cfg, device_map="auto"
    )
    m = PeftModel.from_pretrained(base, adapter_dir)
    m.eval()
    return tok, m


def simulate_customer(history: str, agent_action: str, tok, m, max_new=80):
    prompt = (
        f"[HISTORY]\n{history}\n"
        f"[AGENT ACTION]\n{agent_action}\n"
        f"[CUSTOMER RESPONSE]\n"
    )
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.eos_token_id,
        )
    decoded = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()


# Load the adapter
sim_tok, sim_model = load_simulator(BASE_MODEL, ADAPTER_DIR)
print("Simulator loaded.")

# Test it
test_history = "Agent: Hi! How can I help you today?"
test_action  = "Agent: Sure, can I have your name please?"

response = simulate_customer(test_history, test_action, sim_tok, sim_model)
print("Simulated customer:", response)

### 7B. Ollama Inference (quick test, no adapter)

In [ ]:
from ollama import chat

OLLAMA_MODEL = "gemma4:e4b"

def ollama_simulate(history: str, agent_action: str) -> str:
    system_msg = (
        "You are a customer in a customer-service conversation. "
        "Given the conversation history and the latest agent action, "
        "respond as the customer would — naturally, concisely, and in character."
    )
    user_msg = (
        f"[HISTORY]\n{history}\n"
        f"[AGENT ACTION]\n{agent_action}\n"
        f"[CUSTOMER RESPONSE]"
    )
    response = chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ]
    )
    return response.message.content


history = "Agent: Hi! How can I help you today?"
action  = "Agent: Sure, can I have your name please?"

print("Ollama simulator response:")
print(ollama_simulate(history, action))

## 8. (Optional) Export adapter for use with Ollama Modelfile

If you want to merge the LoRA weights into a full model and load it via Ollama:

```python
merged = sim_model.merge_and_unload()
merged.save_pretrained("./gemma_simulator_merged")
sim_tok.save_pretrained("./gemma_simulator_merged")
# Then: ollama create customer-sim -f Modelfile
```

Modelfile example:
```
FROM ./gemma_simulator_merged
SYSTEM "You are a customer simulator for a retail customer-service RL environment."
```